# Prototype Colab — FR → Wolof TTS (hybride)
Ce notebook fonctionne avec ou sans secrets (mode fallback mock).

In [ ]:
!pip install -q -r /content/Anansi-AI/requirements.txt || true
import os
print('Dépendances installées (ou partiellement en fallback).')

In [ ]:
import os
USE_HF_DATASET = os.getenv('USE_HF_DATASET', 'true').lower() == 'true'
print('USE_HF_DATASET =', USE_HF_DATASET)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Drive non monté:', e)

In [ ]:
from src.translate import translate_text
print('Exemple FR→WO:', translate_text('Bonjour', target_lang='wo'))

In [ ]:
from datasets import load_dataset
import os
dataset_loaded = False
if USE_HF_DATASET:
    try:
        ds = load_dataset('galsenai/anta_women_tts')
        dataset_loaded = True
        print(ds)
    except Exception as e:
        print('Dataset HF inaccessible sans token/permission:', e)
        print('Définir HUGGINGFACE_TOKEN si nécessaire.')
else:
    print('Chargement HF désactivé (USE_HF_DATASET=false).')

In [ ]:
!python src/prepare_dataset.py --input_dir dataset --output_dir dataset_ready
!python src/train.py --dataset_dir dataset_ready --output_dir finetuned --epochs 1 --backend xtts
!mkdir -p outputs
!python src/synthesize.py --text "Nanga def" --model_path finetuned/checkpoint_mock.json --output outputs/sample1.wav
!python src/synthesize.py --text "Mangi fi rek" --model_path finetuned/checkpoint_mock.json --output outputs/sample2.wav
!python src/synthesize.py --text "Jërëjëf" --model_path finetuned/checkpoint_mock.json --output outputs/sample3.wav
print('WAVs générés dans outputs/.')